# Module 4: Hands-On - RNN and LSTM Architectures

## Objectives:
## Objectives:
1. Understand the architecture of RNN and LSTM networks.
2. Explore the outputs of RNN hidden states and LSTM gate activations.
3. Compare RNN and LSTM on handling sequential dependencies.

Tokenization and Preprocessing

In [1]:
# Sample SMS message from SMS Spam dataset
sms_message = "Free entry in a weekly competition to win FA Cup final tickets. Text FA to 12345."

# Simple preprocessing: Tokenization and integer encoding
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts([sms_message])
sequence = tokenizer.texts_to_sequences([sms_message])[0]

print("Original SMS Message:", sms_message)
print("Tokenized Sequence:", sequence)

2026-02-08 09:06:09.428204: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-08 09:06:09.838921: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-08 09:06:11.269568: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Original SMS Message: Free entry in a weekly competition to win FA Cup final tickets. Text FA to 12345.
Tokenized Sequence: [3, 4, 5, 6, 7, 8, 1, 9, 2, 10, 11, 12, 13, 2, 1, 14]


RNN Hidden State Output

In [2]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Embedding

# Parameters
timesteps = len(sequence)
features = 1

# Prepare data
x_data = np.expand_dims(sequence, axis=0)  # Batch size = 1
x_data = np.expand_dims(x_data, axis=-1)   # Add feature dimension

# Build RNN model
model = Sequential([
    SimpleRNN(8, activation='tanh', input_shape=(timesteps, features), return_sequences=True)
])

# Initialize model
model.compile(optimizer='adam', loss='mse')

# Get RNN outputs
rnn_output = model.predict(x_data)
print("RNN Hidden States Shape:", rnn_output.shape)
print("RNN Hidden States:", rnn_output[0])  # Outputs for each timestep

I0000 00:00:1770559573.263957    4394 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13495 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4080 SUPER, pci bus id: 0000:01:00.0, compute capability: 8.9
/home/water/Grad School/DSE6212/.venv/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 273ms/step

2026-02-08 09:06:14.210353: I external/local_xla/xla/service/service.cc:163] XLA service 0x717c100d19d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-02-08 09:06:14.210401: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4080 SUPER, Compute Capability 8.9
2026-02-08 09:06:14.226118: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-02-08 09:06:14.249627: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91801
I0000 00:00:1770559574.408516    4550 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 282ms/step
RNN Hidden States Shape: (1, 16, 8)
RNN Hidden States: [[-0.8919003  -0.94482267 -0.87452304  0.83416754  0.22101769  0.39171422
  -0.75654536  0.68373865]
 [-0.95072633 -0.95177007 -0.57068056  0.8999597   0.74573857  0.40923527
  -0.9621873   0.986195  ]
 [-0.98156023 -0.9803091  -0.7848605   0.9822069   0.8650107   0.7426373
  -0.97612673  0.99410063]
 [-0.9866994  -0.99458563 -0.8922588   0.9919566   0.8986201   0.7687082
  -0.98767877  0.9979405 ]
 [-0.994207   -0.9983134  -0.9519463   0.9958725   0.9187756   0.819563
  -0.99346083  0.9989425 ]
 [-0.997482   -0.99949163 -0.97946525  0.9980647   0.93300974  0.8563372
  -0.9966053   0.99943876]
 [ 0.03947952  0.01091759  0.70642954  0.5801735   0.8241144   0.29662442
  -0.7075009   0.9737808 ]
 [-0.999766   -0.9999659  -0.99924916  0.9998515   0.86128247  0.9687226
  -0.9962113   0.9969482 ]
 [-0.3653089  -0.5756135   0.4206169   0.79027265  0.8431059   0.34723532
  -0.8505438   0.985529  ]
 [-

LSTM Gate Outputs

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import LSTM, Input, RNN
from tensorflow.keras.models import Model

# Custom LSTM cell to extract intermediate gate outputs
class LSTMWithGateOutputs(tf.keras.layers.LSTMCell):
    def call(self, inputs, states, training=None):
        # Get the internal gate computations
        h_tm1 = states[0]  # previous memory state
        c_tm1 = states[1]  # previous carry state

        # Extract the gate computations from the LSTMCell's internal method
        z = tf.keras.backend.dot(inputs, self.kernel)
        z += tf.keras.backend.dot(h_tm1, self.recurrent_kernel)
        z = tf.keras.backend.bias_add(z, self.bias)

        z0, z1, z2, z3 = tf.split(z, 4, axis=1)

        i = self.recurrent_activation(z0)
        f = self.recurrent_activation(z1)
        c = self.activation(z2)
        o = self.recurrent_activation(z3)

        # Calculate new cell state and output
        c = f * c_tm1 + i * c
        h = o * self.activation(c)

        # Return the output and the new states, along with the gate outputs
        return [h, f, i, o], [h, c]

# Define input sequence
timesteps = len(sequence)
features = 1

x_data = np.expand_dims(sequence, axis=0)  # Batch size = 1
x_data = np.expand_dims(x_data, axis=-1)  # Add feature dimension

# Create input and LSTM layer with custom outputs
input_layer = Input(shape=(timesteps, features))
# Initialize the LSTM cell without return_sequences and return_state
lstm_cell = LSTMWithGateOutputs(units=8, activation='tanh')
# Create an RNN layer using the custom cell, and specify return_sequences and return_state here
rnn_layer = tf.keras.layers.RNN(
    lstm_cell, return_sequences=True, return_state=True
)
# Get outputs from the RNN layer
outputs = rnn_layer(input_layer)

# Unpack the outputs - hidden states, ft, it, ot, final_h, final_c
lstm_layer = outputs[0]  # Hidden states for each timestep
final_h = outputs[1]    # Final hidden state
final_c = outputs[2]    # Final cell state

# The intermediate gate outputs (ft, it, ot) are now directly available
ft = outputs[0][:, :, 0:8]  # Assuming 8 units, the gate outputs are in the sequence
it = outputs[0][:, :, 8:16] # Assuming 8 units, the gate outputs are in the sequence
ot = outputs[0][:, :, 16:24] # Assuming 8 units, the gate outputs are in the sequence

# Build model
model = Model(inputs=input_layer, outputs=[lstm_layer, ft, it, ot, final_h, final_c])
model.compile(optimizer='adam', loss='mse')

# Get outputs
lstm_hidden_states, ft_output, it_output, ot_output, final_hidden, final_cell = model.predict(x_data)

# Display results
print("Forget Gate Outputs (ft):", ft_output)
print("\nInput Gate Outputs (it):", it_output)
print("\nOutput Gate Outputs (ot):", ot_output)
print("\nFinal Hidden State:", final_hidden)
print("\nFinal Cell State:", final_cell)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 240ms/step
Forget Gate Outputs (ft): [[[[-0.14558269 -0.18722558  0.11894389  0.0764181   0.05459307
    -0.12651229  0.00521974  0.23084275]
   [-0.3124112  -0.35167006  0.11757082  0.08043065  0.05852285
    -0.16878082  0.02780827  0.41484272]
   [-0.4533628  -0.46854627  0.09134398  0.0587355   0.04660477
    -0.16031833  0.04520399  0.54666305]
   [-0.5514945  -0.54316217  0.0663997   0.03698408  0.03262623
    -0.13202278  0.05260479  0.63799214]
   [-0.61406326 -0.59064215  0.0468359   0.02205478  0.02108611
    -0.10216765  0.05276702  0.7003484 ]
   [-0.6534881  -0.62257886  0.03242352  0.01305793  0.01292199
    -0.07770835  0.04974607  0.7426484 ]
   [-0.5416604  -0.52361244  0.35675088  0.04452856  0.08998123
    -0.15440483  0.11485665  0.4460954 ]
   [-0.63631475 -0.6095832   0.01984621  0.009738    0.00663485
    -0.06404664  0.04776694  0.7166582 ]]]


 [[[ 0.80190974  0.88459617  0.67539227  0.44287562  0.56815666
     0.5350258   0.44350567

## Reflection Questions:
1. How do RNN and LSTM architectures differ in handling long-term dependencies?
2. What insights can you gain from visualizing hidden states and gate activations?
3. Why might LSTMs be better suited for tasks with longer sequences? Provide examples.